# IE4483: product-review sentiment prototype

**NLP task:** review text → negative (0) or positive (1).

Start with the synthetic demo, then switch to the course dataset. **Demo results are not assignment results.** Read `README.md` for setup and the report checklist. All training runs locally on CPU. Run cells in order; each run receives a new folder so earlier evidence is retained.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display
from sentiment import load_data, classify_text

ROOT = Path.cwd()
assert (ROOT / "sentiment.py").exists(), "Open this notebook from the prototype folder."
USE_DEMO = True
TRAIN = ROOT / "data" / ("demo_train.json" if USE_DEMO else "train.json")
TEST = ROOT / "data" / ("demo_test.json" if USE_DEMO else "test.json")
TEMPLATE = ROOT / "data" / ("demo_submission_template.csv" if USE_DEMO else "submission_template.csv")
# Set TEMPLATE = None only if the course does not provide a template.
RUN = ROOT / "runs" / (("demo-" if USE_DEMO else "course-") + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
SEED = 42
print("SYNTHETIC DEMO" if USE_DEMO else "COURSE DATA", "→", RUN)

## 1. Inspect the labelled data

Check class balance, review lengths and missing/duplicate text. The loader rejects missing text and invalid labels; training removes identical case/whitespace-normalised reviews before splitting. Do not split a review into multiple examples.

In [ ]:
data = load_data(TRAIN, labelled=True)
display(data.head())
display(data.groupby("label").size().rename("review_count").to_frame())
display(data.text.str.split().str.len().describe().rename("words_per_review").to_frame())
print("Duplicate normalised reviews:", data.text.str.lower().duplicated().sum())

## 2. Train and compare

**Baseline:** TF-IDF (V features) → logistic regression → one binary prediction.

**Neural comparison:** TF-IDF (V features) → Dense(64, ReLU) → Dense(1, sigmoid).

Both use the same stratified 80/20 split and fit TF-IDF only on the training portion. The best model by validation macro-F1 is then refit on all unique labelled reviews. The neural model is an MLP, not a contextual transformer. No model is guaranteed to win.

In [ ]:
command = [sys.executable, "sentiment.py", "train", "--train", str(TRAIN),
           "--output", str(RUN), "--model", "both", "--seed", str(SEED)]
completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
completed.check_returncode()
display(pd.read_csv(RUN / "comparison.csv"))

## 3. Examine validation evidence

These results belong to the split-trained models, before the final refit. A confusion matrix has actual classes in rows and predicted classes in columns. Read any convergence warnings before interpreting results.

In [ ]:
for name in ["logistic", "mlp"]:
    metrics = json.loads((RUN / f"{name}_metrics.json").read_text(encoding="utf-8"))
    print(name, "warnings:", metrics["warnings"])
    display(pd.DataFrame(metrics["confusion_matrix_rows_true_columns_predicted"],
                         index=["actual negative", "actual positive"],
                         columns=["predicted negative", "predicted positive"]))
    predictions = pd.read_csv(RUN / f"{name}_validation_predictions.csv")
    print("Incorrect examples")
    display(predictions.loc[~predictions.correct].head(3))
    print("Correct examples")
    display(predictions.loc[predictions.correct].head(2))

## 4. Try your own sentence

Edit the text below. A binary model must choose a side for mixed or neutral wording; the probability is an estimate, not a guarantee. A model trained on the demo has very limited vocabulary and ability.

In [ ]:
sentence = "The screen is beautiful, but the battery is terrible."
classify_text(RUN / "model.joblib", sentence)

## 5. Export predictions

This uses the final refitted model. IDs are checked and aligned to the template; test labels are never used for fitting or model selection. The sample template is intentionally in a different order. Download the real submission template before making a course submission.

In [ ]:
submission = RUN / ("demo_submission.csv" if USE_DEMO else "submission.csv")
command = [sys.executable, "sentiment.py", "predict", "--model-file", str(RUN / "model.joblib"),
           "--test", str(TEST), "--output", str(submission)]
if TEMPLATE is not None:
    command += ["--template", str(TEMPLATE)]
completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
completed.check_returncode()
display(pd.read_csv(submission, dtype={"id": str}))

## 6. Turn this prototype into your experiment

1. Set `USE_DEMO = False`, supply the real files, and rerun from the first code cell.
2. Compare unigrams/bigrams, vocabulary size, logistic regularisation, and MLP hidden units or learning rate. Change one setting at a time.
3. Save each run, explain the selected settings, and repeat the final setup with multiple seeds.
4. Use the rubric checklist in `README.md` to cover every report part, including the hotel-review questions.
5. For a contextual model, extend the pipeline with fine-tuned DistilBERT and compare on the same split. That extension is not implemented in this CPU prototype.

If the supplied test set has no labels, discuss with the lecturer how to fulfil the test error-analysis requirement. Validation examples can support development, but are not secretly labelled test results.